# Práctica en vivo · Sesión 5 — Construir, comparar y evaluar modelos
**Unidad didáctica:** Diseño de Soluciones de Inteligencia Artificial (CERTUS) · AA3 · Temas 8 y 9

## Caso: ¿qué clientes se irán de la operadora móvil?
Usamos el mismo archivo de la práctica de las semanas 3 y 4 (`clientes_telecom.csv`). La variable objetivo es `se_fue` (1 = se fue, 0 = se quedó).
Esta vez **no nos centramos en limpiar datos**, sino en responder una pregunta distinta:

> *Entre varios modelos posibles, ¿cuál elijo, cómo sé que es bueno y cómo sé que no me estoy engañando?*

**Ruta del cuaderno**
1. Preparar los datos (limpieza mínima + pipeline de preprocesamiento).
2. El modelo "tonto" (baseline): el punto de comparación obligatorio.
3. Construir 4 familias de modelos: lineal, árbol, boosting y SVM.
4. Métricas: exactitud, precisión, recall, F1 y AUC.
5. Cross validation: una nota más honesta que un solo examen.
6. Overfitting y underfitting con una curva.
7. Grid search y random search: afinar hiperparámetros.
8. Umbral de decisión: decidir según el costo del negocio.
9. Error analysis: ¿dónde se equivoca el modelo?
10. Evaluación final en `test` (una sola vez) y conclusiones.
11. Bonus: métricas de regresión (MAE, RMSE, R²).

## 0. Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     cross_validate, GridSearchCV, RandomizedSearchCV, validation_curve)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             mean_absolute_error, mean_squared_error, r2_score,
                             precision_recall_curve)
from scipy.stats import randint, uniform

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (7, 4)
SEMILLA = 42

In [ ]:
# Cargamos el archivo desde el repositorio del curso.
# Si no hay internet, sube clientes_telecom.csv a Colab (panel de archivos de la izquierda) y cambia URL por el nombre del archivo.
URL = "https://raw.githubusercontent.com/jeanpier22/Diseno-de-Soluciones-de-Inteligencia-Artificial/main/Practica_Semana3y4/clientes_telecom.csv"
df = pd.read_csv(URL, encoding="utf-8-sig")
print(df.shape)
df.head()

### Limpieza mínima
En las semanas 3 y 4 trabajamos esto a fondo. Aquí solo aplicamos lo indispensable para poder **comparar modelos**:
* `edad` con valores imposibles (0 o 999) → se vuelven nulos.
* `monto_factura` negativo → nulo.
* `tipo_plan` escrito de formas distintas (`Prepago`, `PREPAGO`, `Pre-pago`...) → una sola categoría.
* Los nulos que quedan los resuelve el *pipeline* (imputación), **dentro** de cada partición, para no filtrar información de `test`.

In [ ]:
df = df.drop_duplicates(subset="id_cliente").copy()
df.loc[(df["edad"] < 14) | (df["edad"] > 100), "edad"] = np.nan
df.loc[df["monto_factura"] < 0, "monto_factura"] = np.nan
df["tipo_plan"] = (df["tipo_plan"].str.lower().str.replace("-", "", regex=False)
                   .replace({"prepago": "Prepago", "postpago ilimitado": "Postpago Ilimitado", "postpago básico": "Postpago Básico"}))
print(df["tipo_plan"].value_counts())
print("\nClientes que se fueron:", df["se_fue"].mean().round(3), "(proporción)")

### Separar entrenamiento y prueba
El archivo ya trae la columna `particion`. **`test` se queda guardado hasta el final**: es el examen de verdad. Todo lo demás (comparar, afinar, decidir) se hace solo con `train`.

In [ ]:
train = df[df["particion"] == "train"]
test  = df[df["particion"] == "test"]

num = ["edad", "consumo_datos_mb", "minutos_llamadas", "recargas_mes", "monto_factura",
       "velocidad_mbps", "caidas_red_mes", "llamadas_soporte", "satisfaccion"]
cat = ["tipo_plan", "metodo_pago", "departamento"]

X_train, y_train = train[num + cat], train["se_fue"]
X_test,  y_test  = test[num + cat],  test["se_fue"]
print("train:", X_train.shape, " test:", X_test.shape)

prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
])

## 1. El modelo "tonto" (baseline)
Antes de presumir un modelo, hay que saber **qué tan bien lo haría alguien que no sabe nada**. El `DummyClassifier` siempre responde la clase más común.

In [ ]:
dummy = Pipeline([("prep", prep), ("modelo", DummyClassifier(strategy="most_frequent"))]).fit(X_train, y_train)
pred = dummy.predict(X_train)
print("Exactitud del modelo tonto:", round(accuracy_score(y_train, pred), 3))
print("Recall de los que se van   :", recall_score(y_train, pred))

> **Lectura:** el modelo tonto acierta ~71 % **sin detectar a ni un solo cliente que se va**. Por eso la exactitud *sola* engaña cuando las clases están desbalanceadas. Cualquier modelo real debe superar este piso **en las métricas que importan**, no solo en exactitud.

## 2. Construir 4 familias de modelos
Mismo preprocesamiento, mismos datos; cambia solo el algoritmo:

| Familia | Modelo | Idea en una frase |
|---|---|---|
| Lineal | `LogisticRegression` | Suma ponderada de las variables → probabilidad |
| Árbol | `DecisionTreeClassifier` | Cadena de preguntas "si... entonces..." |
| Boosting | `GradientBoostingClassifier` | Muchos árboles pequeños, cada uno corrige los errores del anterior |
| SVM | `SVC` | Busca la frontera con el mayor margen entre clases |

In [ ]:
modelos = {
    "Regresión logística": LogisticRegression(max_iter=1000),
    "Árbol de decisión":   DecisionTreeClassifier(random_state=SEMILLA),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=SEMILLA),
    "SVM (RBF)":           SVC(probability=True, random_state=SEMILLA),
}
pipes = {n: Pipeline([("prep", prep), ("modelo", m)]) for n, m in modelos.items()}
for n, p in pipes.items():
    p.fit(X_train, y_train)
    print("entrenado:", n)

## 3. Métricas de rendimiento
Para clasificación, todas salen de la **matriz de confusión** (VP, FP, FN, VN):
* **Exactitud** = aciertos / total.
* **Precisión** = de los que dije "se va", ¿cuántos realmente se fueron?
* **Recall** = de los que realmente se fueron, ¿a cuántos detecté?
* **F1** = equilibrio entre precisión y recall.
* **AUC** = qué tan bien ordena el modelo a los clientes de menor a mayor riesgo (1 = perfecto, 0,5 = azar).

Aquí medimos **sobre `train` solo con un hold-out interno** (validación), nunca sobre `test`.

In [ ]:
# Hold-out interno: parte del train se reserva como "validación"
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=SEMILLA)

filas = []
for n, p in pipes.items():
    p.fit(X_tr, y_tr)
    pred = p.predict(X_val)
    proba = p.predict_proba(X_val)[:, 1]
    filas.append({"modelo": n,
                  "exactitud": accuracy_score(y_val, pred),
                  "precisión": precision_score(y_val, pred),
                  "recall": recall_score(y_val, pred),
                  "F1": f1_score(y_val, pred),
                  "AUC": roc_auc_score(y_val, proba)})
tabla = pd.DataFrame(filas).set_index("modelo").round(3)
tabla

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (n, p) in zip(ejes, pipes.items()):
    ConfusionMatrixDisplay.from_predictions(y_val, p.predict(X_val), ax=ax, colorbar=False,
                                            display_labels=["Se queda", "Se va"], cmap="Blues")
    ax.set_title(n, fontsize=10)
plt.tight_layout(); plt.show()

> **Para discutir en clase:** ¿qué modelo detecta a más clientes que se van (recall)? ¿Cuál se equivoca menos cuando dice "se va" (precisión)? ¿Es el mismo? **Casi nunca lo es**: elegir un modelo es elegir qué error te duele menos.

## 4. Cross validation: una nota más honesta
Una sola partición puede tener suerte o mala suerte. Con **5 pliegues** el modelo se entrena y evalúa 5 veces, cada vez con un pedazo distinto como examen. Reportamos **promedio ± desviación**.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
resumen = []
for n, p in pipes.items():
    r = cross_validate(p, X_train, y_train, cv=cv, scoring=["roc_auc", "recall", "f1"], n_jobs=-1)
    resumen.append({"modelo": n,
                    "AUC (media)": r["test_roc_auc"].mean(), "AUC (±)": r["test_roc_auc"].std(),
                    "recall (media)": r["test_recall"].mean(), "F1 (media)": r["test_f1"].mean()})
cv_tabla = pd.DataFrame(resumen).set_index("modelo").round(3)
cv_tabla

In [ ]:
cv_tabla["AUC (media)"].sort_values().plot.barh(color="#0D2D6C", xerr=cv_tabla["AUC (±)"].reindex(cv_tabla["AUC (media)"].sort_values().index))
plt.xlim(0.5, 1); plt.xlabel("AUC en validación cruzada (5 pliegues)"); plt.title("Comparación honesta de modelos"); plt.show()

> **Regla práctica:** si la diferencia entre dos modelos es menor que la desviación (±), **no hay un ganador claro**: elige el más simple, rápido o explicable.

## 5. Overfitting y underfitting
La **complejidad** de un árbol se controla con `max_depth`:
* Muy poca profundidad → **underfitting** (el modelo es demasiado simple, falla también en entrenamiento).
* Demasiada profundidad → **overfitting** (memoriza el entrenamiento y falla con datos nuevos).

Señal de alarma: la nota de *train* sube y sube, pero la de *validación* se estanca o baja.

In [ ]:
profundidades = range(1, 16)
pipe_arbol = Pipeline([("prep", prep), ("modelo", DecisionTreeClassifier(random_state=SEMILLA))])
tr, va = validation_curve(pipe_arbol, X_train, y_train, param_name="modelo__max_depth",
                          param_range=profundidades, cv=cv, scoring="roc_auc", n_jobs=-1)
plt.plot(profundidades, tr.mean(axis=1), "o-", label="Entrenamiento", color="#E0245E")
plt.plot(profundidades, va.mean(axis=1), "o-", label="Validación (CV)", color="#0D2D6C")
mejor = list(profundidades)[va.mean(axis=1).argmax()]
plt.axvline(mejor, ls="--", color="gray"); plt.text(mejor + 0.2, 0.55, f"mejor ≈ {mejor}")
plt.xlabel("max_depth (complejidad del árbol)"); plt.ylabel("AUC"); plt.legend(); plt.title("Underfitting → buen punto → Overfitting")
plt.show()
print("Mejor profundidad según validación:", mejor)

## 6. Afinar hiperparámetros: Grid Search y Random Search
Un **hiperparámetro** se fija *antes* de entrenar (profundidad, número de árboles, tasa de aprendizaje...). Dos formas de buscar:
* **Grid search:** prueba *todas* las combinaciones de una cuadrícula (exhaustivo, caro).
* **Random search:** prueba combinaciones *al azar* (más barato, suele encontrar algo igual de bueno).

In [ ]:
pipe_gb = Pipeline([("prep", prep), ("modelo", GradientBoostingClassifier(random_state=SEMILLA))])

rejilla = {"modelo__n_estimators": [50, 100, 200],
           "modelo__learning_rate": [0.03, 0.1, 0.3],
           "modelo__max_depth": [2, 3]}
grid = GridSearchCV(pipe_gb, rejilla, cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
print("GRID   → mejor AUC CV:", round(grid.best_score_, 4), "con", grid.best_params_)
print("combinaciones probadas:", len(grid.cv_results_["params"]))

In [ ]:
espacio = {"modelo__n_estimators": randint(50, 300),
           "modelo__learning_rate": uniform(0.01, 0.3),
           "modelo__max_depth": randint(2, 5),
           "modelo__subsample": uniform(0.6, 0.4)}
rand = RandomizedSearchCV(pipe_gb, espacio, n_iter=15, cv=cv, scoring="roc_auc", n_jobs=-1, random_state=SEMILLA).fit(X_train, y_train)
print("RANDOM → mejor AUC CV:", round(rand.best_score_, 4), "con", {k.split('__')[1]: round(v, 3) for k, v in rand.best_params_.items()})
print("combinaciones probadas:", 15)

> Compara: ¿la búsqueda aleatoria con 15 intentos se acerca a la exhaustiva con 18 combinaciones? En espacios grandes, el random search gana por mucho en tiempo.

**¿Y la regresión logística?** También tiene un hiperparámetro (`C`, la fuerza de la regularización). Le damos la misma oportunidad para que la comparación sea justa.

In [ ]:
pipe_lr = Pipeline([("prep", prep), ("modelo", LogisticRegression(max_iter=1000))])
grid_lr = GridSearchCV(pipe_lr, {"modelo__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]}, cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
print("LOGÍSTICA → mejor AUC CV:", round(grid_lr.best_score_, 4), "con", grid_lr.best_params_)

candidatos = {"Logística afinada": grid_lr, "Boosting (grid)": grid, "Boosting (random)": rand}
for n, g in candidatos.items():
    print(f"{n:20s} AUC CV = {g.best_score_:.4f}")
nombre_elegido = max(candidatos, key=lambda n: candidatos[n].best_score_)
elegido = candidatos[nombre_elegido].best_estimator_
print("\n>>> Modelo elegido:", nombre_elegido)

> **Decisión:** elegimos por la nota de validación cruzada, **no por la fama del algoritmo**. Si un modelo simple empata o gana, nos quedamos con el simple: es más barato, más rápido y más fácil de explicar al negocio.

## 7. Umbral de decisión: decidir según el costo del negocio
El modelo entrega una **probabilidad**; tú decides desde qué valor llamas "se va". Por defecto es 0,5, pero no tiene por qué ser el mejor.

Supongamos: **retener** a un cliente con una oferta cuesta S/ 20, y **perder** a un cliente cuesta S/ 300.
Perder un cliente que se va (falso negativo) es 15 veces más caro que molestar a uno que no se iba (falso positivo).

In [ ]:
mejor_modelo = elegido
mejor_modelo.fit(X_tr, y_tr)
proba = mejor_modelo.predict_proba(X_val)[:, 1]

COSTO_OFERTA, COSTO_PERDER = 20, 300
umbrales = np.arange(0.05, 0.95, 0.05)
costos, recalls, precis = [], [], []
for u in umbrales:
    pred = (proba >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    costos.append(fp * COSTO_OFERTA + tp * COSTO_OFERTA + fn * COSTO_PERDER)
    recalls.append(recall_score(y_val, pred)); precis.append(precision_score(y_val, pred, zero_division=0))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(umbrales, recalls, label="Recall", color="#E0245E"); ax[0].plot(umbrales, precis, label="Precisión", color="#0D2D6C")
ax[0].set_xlabel("Umbral"); ax[0].legend(); ax[0].set_title("Precisión vs recall")
ax[1].plot(umbrales, costos, color="#0D2D6C"); u_opt = umbrales[int(np.argmin(costos))]
ax[1].axvline(u_opt, ls="--", color="gray"); ax[1].set_xlabel("Umbral"); ax[1].set_ylabel("Costo total (S/)"); ax[1].set_title(f"Costo mínimo con umbral ≈ {u_opt:.2f}")
plt.tight_layout(); plt.show()
print("Costo con umbral 0.50:", costos[list(umbrales.round(2)).index(0.5)], "| costo con umbral óptimo:", min(costos))

## 8. Error analysis: ¿dónde se equivoca el modelo?
Una métrica global esconde patrones. Miramos **quiénes** son los errores, sobre todo los **falsos negativos** (clientes que se fueron y no los vimos).

In [ ]:
val = X_val.copy()
val["real"] = y_val.values
val["proba"] = proba
val["pred"] = (proba >= 0.5).astype(int)
val["error"] = np.where((val["real"] == 1) & (val["pred"] == 0), "Falso negativo",
                np.where((val["real"] == 0) & (val["pred"] == 1), "Falso positivo", "Acierto"))
print(val["error"].value_counts(), "\n")
# ¿En qué tipo de plan fallamos más con los que realmente se fueron?
se_fueron = val[val["real"] == 1]
print((se_fueron.groupby("tipo_plan")["pred"].apply(lambda s: (s == 1).mean()).round(2)).rename("recall por plan"))

In [ ]:
fn = val[val["error"] == "Falso negativo"]
vp = val[(val["real"] == 1) & (val["pred"] == 1)]
comparar = pd.DataFrame({"Falsos negativos": fn[["satisfaccion", "caidas_red_mes", "llamadas_soporte", "velocidad_mbps"]].mean(),
                         "Verdaderos positivos": vp[["satisfaccion", "caidas_red_mes", "llamadas_soporte", "velocidad_mbps"]].mean()}).round(2)
comparar

> **Pregunta guía:** ¿en qué se parecen los clientes que el modelo *no* logra ver? Esa es la pista de qué variable nueva crear o qué dato recolectar para la siguiente versión (esto es "descripción de mejoras posibles" de la Evidencia 3).

## 9. Evaluación final en `test` (una sola vez)
Elegimos el modelo y los hiperparámetros **solo con `train`**. Ahora, un único examen final con `test`, que nadie ha visto.

In [ ]:
final = elegido.fit(X_train, y_train)
pred_t = final.predict(X_test)
proba_t = final.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred_t, target_names=["Se queda", "Se va"]))
print("AUC en test:", round(roc_auc_score(y_test, proba_t), 3))
ConfusionMatrixDisplay.from_predictions(y_test, pred_t, display_labels=["Se queda", "Se va"], cmap="Blues")
plt.title("Matriz de confusión · test"); plt.show()

## 10. Bonus: métricas de regresión
Si el objetivo es un **número** (no una clase), cambian las métricas. Predecimos `monto_factura` (S/).
* **MAE:** error promedio, en soles.
* **RMSE:** como el MAE pero castiga más los errores grandes.
* **R²:** qué parte de la variabilidad explica el modelo (1 = todo, 0 = igual que predecir el promedio).

In [ ]:
reg = df.dropna(subset=["monto_factura"]).copy()
reg = reg[reg["monto_factura"] < reg["monto_factura"].quantile(0.99)]   # sin facturas absurdas
Xr, yr = reg[num + cat].drop(columns=["monto_factura"]), reg["monto_factura"]
num_r = [c for c in num if c != "monto_factura"]
prep_r = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num_r),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat)])
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.25, random_state=SEMILLA)

for nombre, m in {"Regresión lineal": LinearRegression(),
                  "Árbol (prof. 4)": DecisionTreeRegressor(max_depth=4, random_state=SEMILLA),
                  "Random Forest": RandomForestRegressor(n_estimators=150, random_state=SEMILLA, n_jobs=-1)}.items():
    p = Pipeline([("prep", prep_r), ("modelo", m)]).fit(Xr_tr, yr_tr)
    y_hat = p.predict(Xr_te)
    print(f"{nombre:18s} MAE = S/ {mean_absolute_error(yr_te, y_hat):6.2f} | RMSE = S/ {np.sqrt(mean_squared_error(yr_te, y_hat)):6.2f} | R² = {r2_score(yr_te, y_hat):.3f}")
print("\nReferencia: predecir siempre el promedio daría MAE =", round(mean_absolute_error(yr_te, np.full(len(yr_te), yr_tr.mean())), 2))

## Ejercicios para entregar (Evidencia 3 · Parte III)
Trabajen con **su caso del proyecto** o con este dataset. Respondan en un informe breve (con capturas):

1. **Baseline:** ¿cuánto logra el modelo tonto y por qué su exactitud engaña?
2. **Comparación:** tabla con al menos 3 modelos, con exactitud, recall, F1 y AUC *en validación cruzada*.
3. **Diagnóstico:** ¿algún modelo muestra overfitting? Muestren la evidencia (curva o diferencia train vs validación).
4. **Afinamiento:** un grid o random search sobre el modelo elegido. ¿Cuánto mejoró?
5. **Negocio:** propongan costos propios para falso positivo y falso negativo y elijan el umbral.
6. **Error analysis:** describan el perfil de los falsos negativos y **dos mejoras** concretas.
7. **Test final:** reporten el resultado una sola vez y concluyan si el modelo está listo para llevarse a una API (Sesión 6).